# 06 - Explainability

SHAP and DiCE explanations for model interpretability.

**Note:** SHAP analysis is performed on the XGBoost base learner, not the full stacking ensemble.

In [ ]:
import sys
sys.path.insert(0, '..')

import joblib
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt

from src.data.load_data import load_and_merge
from src.data.preprocess import run_full_preprocessing_pipeline
from src.features.feature_engineering import create_features
from src.explainability.shap_explainer import explain_xgboost_from_stack
from src.explainability.dice_explainer import setup_dice, generate_counterfactuals

In [ ]:
# Load and preprocess data
df = load_and_merge('../data/raw')
df = create_features(df)

X_train, X_test, y_train, y_test, pipeline, _, _ = run_full_preprocessing_pipeline(df)

# Load trained models
stack_model = joblib.load('../models/stacking_model.pkl')
xgb_model = joblib.load('../models/xgboost_model.pkl')

## SHAP Analysis (XGBoost Base Learner)

SHAP provides global feature importance and explains individual predictions.

In [ ]:
# Generate SHAP explanation for XGBoost
explain_xgboost_from_stack(stack_model, X_test)

In [ ]:
# Generate SHAP explanation for XGBoost directly
explainer = shap.Explainer(xgb_model)
shap_values = explainer(X_test)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, show=False)
plt.title('SHAP Feature Importance (XGBoost Base Learner)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/shap_summary_xgboost.png', dpi=300, bbox_inches='tight')
plt.show()

## SHAP Feature Importance (Bar Plot)

In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, plot_type='bar', show=False)
plt.title('SHAP Feature Importance - Bar (XGBoost Base Learner)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/shap_importance_xgboost.png', dpi=300, bbox_inches='tight')
plt.show()

## DiCE Counterfactual Explanations

Generates hypothetical counterfactual examples showing which features
would need to change for a different prediction.

**Disclaimer:** Counterfactuals are model-generated hypothetical changes,
NOT medical advice. They do not imply that changing a feature will
actually improve a patient's health.

In [ ]:
# Select a patient with predicted disease
patient_idx = X_test.index[0]
patient = X_test.iloc[[0]]
patient_prediction = xgb_model.predict(patient)[0]
patient_proba = xgb_model.predict_proba(patient)[0]

print(f'Patient prediction: {"Disease" if patient_prediction == 1 else "No Disease"}')
print(f'Probability (No Disease): {patient_proba[0]:.3f}')
print(f'Probability (Disease): {patient_proba[1]:.3f}')

In [ ]:
# Set up DiCE
dice_exp = setup_dice(X_train, y_train, xgb_model)

# Generate counterfactuals
counterfactuals = generate_counterfactuals(
    dice_exp, patient, num_counterfactuals=3, desired_class=0
)

# Display counterfactuals
counterfactuals.visualize_as_dataframe(show_only_changes=True)